# OV7670 capture: tampilkan buff_0 dan buff_1 dari DDR3

1. Program device dan tunggu DDR3 terkalibrasi serta buffer berisi frame.
2. Di **Vivado Hardware Manager → Tcl Console**, jalankan (tanpa parameter):
   ```tcl
   source <path>/verifications/capture_frame.tcl
   ```
   Skrip membaca `buff_0` (`0x00000000`) dan `buff_1` (`0x00096000`) lalu menyimpan `buff_0_axi32.hex` dan `buff_1_axi32.hex` di folder `verifications`.
3. Setelah pesan **DONE**, jalankan seluruh cell notebook ini (**Run All**).

Setiap baris HEX berisi satu word 32-bit dalam urutan alamat naik: halfword rendah adalah piksel lebih awal, lalu halfword tinggi. Satu piksel RGB444 disimpan sebagai `0x0RGB` (640×480). Buffer yang file-nya belum ada dilewati.

**Writer tetap berjalan selama dump**, jadi gambar bisa mencampur beberapa frame dan kedua buffer tidak diambil serentak.

Dependensi: `numpy`, `matplotlib`, `Pillow`, dan `IPython` pada kernel notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import re
import warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display


def find_verifications_dir():
    """Folder yang berisi capture_frame.tcl, dari cwd atau induknya."""
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "verifications", parent / "video-streaming" / "verifications"):
            if (candidate / "capture_frame.tcl").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Buka notebook dari folder video-streaming atau verifications.")


def load_hex(path, width, height):
    """Baca dump capture_frame.tcl dan kembalikan piksel RGB444 serta RGB888."""
    path = Path(path).expanduser().resolve()
    expected_words = width*height//2
    values = []
    for line_number, line in enumerate(path.read_text(encoding="ascii").splitlines(), 1):
        text = line.strip()
        if not text or text.startswith("#"):
            continue
        if not re.fullmatch(r"[0-9a-fA-F]{8}", text):
            raise ValueError(f"{path.name}:{line_number}: harus tepat delapan digit hex.")
        values.append(int(text, 16))
    if len(values) != expected_words:
        raise ValueError(f"{path.name}: perlu {expected_words} word untuk {width}x{height}, ditemukan {len(values)}.")
    words = np.asarray(values, dtype=np.uint32)
    if np.any(words == 0xDEC0DEE3):
        raise ValueError(f"{path.name}: ditemukan pola decode-error DEC0DEE3; periksa address map DDR.")
    pixels = np.empty(width*height, dtype=np.uint16)
    pixels[0::2] = words & 0xFFFF
    pixels[1::2] = words >> 16
    bad_padding = int(np.count_nonzero(pixels & 0xF000))
    if bad_padding:
        warnings.warn(f"{path.name}: {bad_padding} piksel memiliki nibble padding atas bukan nol; periksa format/data DDR.")
    rgb = np.stack(((pixels >> 8) & 15, (pixels >> 4) & 15, pixels & 15), axis=-1)
    rgb = (rgb*17).astype(np.uint8).reshape(height, width, 3)
    modified = datetime.fromtimestamp(path.stat().st_mtime, timezone.utc)
    return {"source": path, "pixels": pixels.reshape(height, width), "rgb": rgb,
            "info": {"file": str(path), "modified_utc": modified.strftime("%Y-%m-%dT%H:%M:%SZ"),
                     "word_count": len(values), "pixel_count": int(pixels.size),
                     "nonzero_padding_pixels": bad_padding,
                     "all_black": bool(np.all((pixels & 0x0FFF) == 0))}}


## Muat file HEX

File dicari otomatis di folder `verifications` (tempat `capture_frame.tcl`).

In [ ]:
VERIFICATIONS_DIR = find_verifications_dir()
HEX_FILES = {buffer_id: VERIFICATIONS_DIR / f"buff_{buffer_id}_axi32.hex" for buffer_id in (0, 1)}
WIDTH, HEIGHT = 640, 480
SAVE_PNG = True

frames = {}
for buffer_id, filename in HEX_FILES.items():
    if not Path(filename).expanduser().is_file():
        print(f"buff_{buffer_id}: file belum ada, dilewati -> {filename}")
        continue
    frames[buffer_id] = load_hex(filename, WIDTH, HEIGHT)
    print(f"buff_{buffer_id}: {frames[buffer_id]['info']['file']} (diubah {frames[buffer_id]['info']['modified_utc']})")
if not frames:
    raise FileNotFoundError("Tidak ada file HEX yang ditemukan. Jalankan capture_frame.tcl di Vivado sampai DONE.")


## Gambar

Konversi kanal 4-bit ke 8-bit memakai perkalian 17 (`0 → 0`, `15 → 255`). Tidak ada flip atau pembalikan word tambahan; `capture_frame.tcl` sudah menulis word dalam urutan alamat naik.

In [ ]:
fig, axes = plt.subplots(1, len(frames), figsize=(7*len(frames), 6), squeeze=False)
for ax, (buffer_id, frame) in zip(axes[0], sorted(frames.items())):
    ax.imshow(frame["rgb"], interpolation="nearest")
    ax.set_title(f"buff_{buffer_id} | {frame['info']['modified_utc']}")
    ax.axis("off")
fig.suptitle(f"{WIDTH} x {HEIGHT} RGB444 | OV7670 capture dari DDR3")
plt.tight_layout()
plt.show()


## Pemeriksaan data dan ekspor PNG

Padding atas yang bukan nol dapat menunjukkan format atau data DDR yang tidak sesuai. Gambar hitam bisa berarti buffer masih nol atau scene memang gelap.

In [ ]:
rows = ["| Buffer | Word 32-bit | Piksel | Padding salah | Semua hitam |",
        "|---|---:|---:|---:|---|"]
for buffer_id, frame in sorted(frames.items()):
    info = frame["info"]
    rows.append(f"| buff_{buffer_id} | {info['word_count']} | {info['pixel_count']} | {info['nonzero_padding_pixels']} | {info['all_black']} |")
display(Markdown("\n".join(rows)))
if len(frames) == 2:
    changed = int(np.count_nonzero(frames[0]["pixels"] != frames[1]["pixels"]))
    print(f"Piksel berbeda antar-buffer: {changed:,} / {WIDTH*HEIGHT:,}.")

if SAVE_PNG:
    for buffer_id, frame in sorted(frames.items()):
        output = frame["source"].with_name(frame["source"].stem.replace("_axi32", "") + "_rgb444.png")
        Image.fromarray(frame["rgb"]).save(output)
        print("PNG:", output)
